# QA: Compare Phase Error Correction Across Two Runs

Side-by-side comparison of two inference runs for every patient.
Each figure is a 9×6 grid: columns 1–3 are **Run A**, columns 4–6 are **Run B**.

| Row | Content | Colormap |
|-----|---------|----------|
| 1   | CNN raw correction (per-timepoint) — vx, vy, vz | jet |
| 2   | CNN polyfit correction (time-independent) — vx, vy, vz | jet |
| 3   | Raw manual correction (corrected vel − uncorrected vel) — vx, vy, vz | jet |
| 4   | Manual polyfit correction (time-independent) — vx, vy, vz | jet |
| 5   | Manual polyfit − CNN polyfit on mediastinum — vx, vy, vz | RdBu_r |
| 6   | CNN corrected velocity (uncorr + polyfit) — vx, vy, vz | RdBu_r |
| 7   | Manually corrected velocity — vx, vy, vz | RdBu_r |
| 8   | Uncorrected velocity — vx, vy, vz | RdBu_r |
| 9   | Magnitude, tissue mask, used pixels | gray / RdBu_r |

Rows 3–4, 7–9 are identical across runs (data-only, no inference dependence).
Row 5 differs per run (residual depends on each run's CNN polyfit).

Images are saved to `./qa_pec_compare_images/`.

In [9]:
import platform
from pathlib import Path

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import zoom

from vascular_superenhancement.utils.path_config import load_path_config, _PROJECT_ROOT

config_name = "local_mac" if platform.system() == "Darwin" else "all_patients"
pc = load_path_config(config_name)

WORKING_DIR = pc.working_dir
PATIENT_DATA_DIR = WORKING_DIR / "patient_data"
REPO_ROOT = _PROJECT_ROOT
DOWNSAMPLED_FOLDER = "downsampled_full_fov_128x128x64"

RUN_A_NAME = "lucky-eon_epoch_68"
RUN_B_NAME = "fearless-paper_epoch-19"  # <-- set the second run name here

INFERENCE_BASE = WORKING_DIR.parent / "inference"
INFERENCE_DIR_A = INFERENCE_BASE / RUN_A_NAME
INFERENCE_DIR_B = INFERENCE_BASE / RUN_B_NAME

OUTPUT_DIR = Path("qa_pec_compare_images") / f'{RUN_A_NAME}_vs_{RUN_B_NAME}'
OUTPUT_DIR.mkdir(exist_ok=True)

SPLITS = ["test"]  # e.g. ["test"], ["train", "validation"], or ["train", "validation", "test"]

FRAME_INDEX = 4

print(f"Patient data dir: {PATIENT_DATA_DIR}")
print(f"Run A ({RUN_A_NAME}): {INFERENCE_DIR_A}")
print(f"Run B ({RUN_B_NAME}): {INFERENCE_DIR_B}")
print(f"Output dir:       {OUTPUT_DIR.resolve()}")

Patient data dir: /Users/yakhilesh/Files/7_PhD/vascular-superenhancement/code-base/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data
Run A (lucky-eon_epoch_68): /Users/yakhilesh/Files/7_PhD/vascular-superenhancement/code-base/vascular-superenhancement-4d-flow/working_dir/inference/lucky-eon_epoch_68
Run B (fearless-paper_epoch-19): /Users/yakhilesh/Files/7_PhD/vascular-superenhancement/code-base/vascular-superenhancement-4d-flow/working_dir/inference/fearless-paper_epoch-19
Output dir:       /Users/yakhilesh/Files/7_PhD/vascular-superenhancement/code-base/vascular-superenhancement-4d-flow/notebooks/data-qa/qa_pec_compare_images/lucky-eon_epoch_68_vs_fearless-paper_epoch-19


In [10]:
splits_df = pd.read_csv(REPO_ROOT / "splits" / "splits_01-15-26.csv")
patients_df = splits_df[splits_df["split"].isin(SPLITS)].copy()
patients_df = patients_df.sort_values(["split", "patient_id"]).reset_index(drop=True)

print(f"Total patients to inspect: {len(patients_df)}")
print(patients_df["split"].value_counts())

Total patients to inspect: 28
split
test    28
Name: count, dtype: int64


In [11]:
COMPONENTS = ["vx", "vy", "vz"]


def load_axial_slice(nifti_path: Path, z_idx: int | None = None) -> np.ndarray:
    """Load a NIfTI file and return an axial slice as a 2-D array."""
    vol = nib.load(str(nifti_path)).get_fdata(dtype=np.float32)
    if z_idx is None:
        z_idx = vol.shape[2] // 2
    return vol[:, :, z_idx]


def load_air_mask_slice(patient_dir: Path, pid: str, ds_shape: tuple,
                        z_idx: int | None = None) -> np.ndarray:
    """Load the correction air mask and return a resampled axial slice."""
    mask_path = patient_dir / "nifti" / "velocity_correction" / f"correction_air_mask_{pid}.nii.gz"
    mask_vol = nib.load(str(mask_path)).get_fdata(dtype=np.float32)
    if mask_vol.shape != ds_shape:
        scale = np.array(ds_shape) / np.array(mask_vol.shape)
        mask_vol = zoom(mask_vol, scale, order=0)
    if z_idx is None:
        z_idx = mask_vol.shape[2] // 2
    return mask_vol[:, :, z_idx] > 0.5


def apply_mask(arr: np.ndarray, mask: np.ndarray) -> np.ma.MaskedArray:
    return np.ma.masked_where(~mask, arr)


def _try_load_axial_slice(nifti_path: Path, z_idx: int | None = None) -> np.ndarray | None:
    if not nifti_path.exists():
        return None
    return load_axial_slice(nifti_path, z_idx)


def _load_run_data(pid: str, frame: int, ds_root: Path, mag_path: Path,
                   inf_root: Path | None, uncorr: dict,
                   z_idx: int | None = None) -> dict:
    """Load CNN predictions and derived quantities for a single run.

    Returns a dict with keys: pred_raw, pred_poly, cnn_corr, and
    corresponding has_* booleans.
    """
    pred_raw = {}
    pred_poly = {}
    if inf_root is not None and inf_root.exists():
        poly_z = None
        if z_idx is not None:
            first_poly = inf_root / "predicted_corrected_velocity" / f"pred_correction_vx_{pid}.nii.gz"
            if first_poly.exists():
                poly_nz = nib.load(str(first_poly)).shape[2]
                ds_nz = nib.load(str(mag_path)).shape[2]
                poly_z = int(round(z_idx / (ds_nz - 1) * (poly_nz - 1)))

        for comp in COMPONENTS:
            pred_raw[comp] = _try_load_axial_slice(
                inf_root / "raw_predictions" / f"pred_correction_{comp}_t{frame:02d}.nii.gz", z_idx
            )
            pred_poly[comp] = _try_load_axial_slice(
                inf_root / "predicted_corrected_velocity" / f"pred_correction_{comp}_{pid}.nii.gz", poly_z
            )

    has_pred_raw = all(pred_raw.get(c) is not None for c in COMPONENTS)
    has_pred_poly = all(pred_poly.get(c) is not None for c in COMPONENTS)

    if (has_pred_raw or has_pred_poly) and inf_root is not None:
        coeff_path = inf_root / "predicted_corrected_velocity" / f"pred_poly_coefficients_{pid}.npz"
        if coeff_path.exists():
            venc = float(np.load(str(coeff_path))["venc"])
            for c in COMPONENTS:
                if pred_raw.get(c) is not None:
                    pred_raw[c] = pred_raw[c] * venc
                if pred_poly.get(c) is not None:
                    pred_poly[c] = pred_poly[c] * venc

    has_uncorr = all(uncorr[c] is not None for c in COMPONENTS)
    cnn_corr = {}
    has_cnn_corr = has_uncorr and has_pred_poly
    if has_cnn_corr:
        for comp in COMPONENTS:
            poly_slice = pred_poly[comp]
            ref_shape = uncorr[comp].shape
            if poly_slice.shape != ref_shape:
                poly_slice = zoom(poly_slice,
                                  np.array(ref_shape) / np.array(poly_slice.shape),
                                  order=1)
            cnn_corr[comp] = uncorr[comp] + poly_slice

    return dict(
        pred_raw=pred_raw, has_pred_raw=has_pred_raw,
        pred_poly=pred_poly, has_pred_poly=has_pred_poly,
        cnn_corr=cnn_corr, has_cnn_corr=has_cnn_corr,
    )


def make_comparison_figure(
    pid: str, split: str, frame: int, ds_root: Path, patient_dir: Path,
    inf_root_a: Path | None, inf_root_b: Path | None,
    run_a_label: str = "Run A", run_b_label: str = "Run B",
    z_idx: int | None = None,
) -> plt.Figure:
    """7-row × 6-column comparison figure. Cols 0–2 = Run A, Cols 3–5 = Run B."""
    mag_path = ds_root / "4d_flow_mag" / f"4d_flow_mag_{pid}_frame_{frame:02d}.nii.gz"
    mag_slice = load_axial_slice(mag_path, z_idx)

    uncorr = {}
    corr = {}
    for comp in COMPONENTS:
        uncorr[comp] = _try_load_axial_slice(
            ds_root / f"4d_flow_{comp}" / f"4d_flow_{comp}_{pid}_frame_{frame:02d}.nii.gz", z_idx
        )
        corr[comp] = _try_load_axial_slice(
            ds_root / f"4d_flow_{comp}_corr" / f"4d_flow_{comp}_corr_{pid}_frame_{frame:02d}.nii.gz", z_idx
        )

    has_uncorr = all(uncorr[c] is not None for c in COMPONENTS)
    has_corr = all(corr[c] is not None for c in COMPONENTS)
    has_diff = has_uncorr and has_corr
    diff = {}
    if has_diff:
        diff = {comp: corr[comp] - uncorr[comp] for comp in COMPONENTS}

    run_a = _load_run_data(pid, frame, ds_root, mag_path, inf_root_a, uncorr, z_idx)
    run_b = _load_run_data(pid, frame, ds_root, mag_path, inf_root_b, uncorr, z_idx)

    ds_shape = nib.load(str(mag_path)).shape
    tissue = load_air_mask_slice(patient_dir, pid, ds_shape, z_idx)
    mediastinum = tissue

    # Manual polyfit correction (time-independent, stored normalised as v/VENC)
    manual_poly = {}
    corr_dir = patient_dir / "nifti" / "velocity_correction"
    ref_shape_2d = mag_slice.shape
    man_venc_path = corr_dir / f"poly_coefficients_{pid}.npz"
    man_venc = float(np.load(str(man_venc_path))["venc"]) if man_venc_path.exists() else 1.0
    for comp in COMPONENTS:
        mp_path = corr_dir / f"ground_truth_correction_{comp}_{pid}.nii.gz"
        if mp_path.exists():
            mp_slice = load_axial_slice(mp_path, z_idx)
            if mp_slice.shape != ref_shape_2d:
                mp_slice = zoom(mp_slice,
                                np.array(ref_shape_2d) / np.array(mp_slice.shape),
                                order=1)
            manual_poly[comp] = mp_slice * man_venc
        else:
            manual_poly[comp] = None
    has_manual_poly = all(manual_poly[c] is not None for c in COMPONENTS)

    # Per-run residual: manual polyfit − CNN polyfit
    for run in (run_a, run_b):
        resid = {}
        has_resid = has_manual_poly and run["has_pred_poly"]
        if has_resid:
            for comp in COMPONENTS:
                cnn_slice = run["pred_poly"][comp]
                man_slice = manual_poly[comp]
                if cnn_slice.shape != man_slice.shape:
                    cnn_slice = zoom(cnn_slice,
                                     np.array(man_slice.shape) / np.array(cnn_slice.shape),
                                     order=1)
                resid[comp] = man_slice - cnn_slice
        run["corr_residual"] = resid
        run["has_corr_residual"] = has_resid

    vel_max = 300.0

    corr_max = 1.0
    corr_vals = []
    for run in (run_a, run_b):
        if run["has_pred_raw"]:
            for c in COMPONENTS:
                corr_vals.append(run["pred_raw"][c].ravel())
        if run["has_pred_poly"]:
            for c in COMPONENTS:
                corr_vals.append(run["pred_poly"][c].ravel())
        if run["has_corr_residual"]:
            for c in COMPONENTS:
                corr_vals.append(run["corr_residual"][c].ravel())
    if has_diff:
        for c in COMPONENTS:
            corr_vals.append(diff[c].ravel())
    if has_manual_poly:
        for c in COMPONENTS:
            corr_vals.append(manual_poly[c].ravel())
    if corr_vals:
        corr_max = np.percentile(np.abs(np.concatenate(corr_vals)), 99)
        if corr_max == 0:
            corr_max = 1.0

    bg_color = "0.25"
    vel_cmap = plt.cm.RdBu_r.copy()
    vel_cmap.set_bad(bg_color)
    jet_cmap = plt.cm.jet.copy()
    jet_cmap.set_bad(bg_color)
    resid_cmap = plt.cm.RdBu_r.copy()
    resid_cmap.set_bad(bg_color)

    z_label = f"z={z_idx}" if z_idx is not None else "z=mid"
    n_rows = 9
    n_cols = 6
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(26, n_rows * 3.5),
                             constrained_layout=True)
    fig.suptitle(f"{pid}  [{split}]  frame {frame:02d}  {z_label}\n"
                 f"{run_a_label}  vs  {run_b_label}",
                 fontsize=16, fontweight="bold")

    runs = [run_a, run_b]
    run_labels = [run_a_label, run_b_label]
    col_offsets = [0, 3]

    for ri, (run, label, col0) in enumerate(zip(runs, run_labels, col_offsets)):
        # Row 0: CNN raw correction — jet, full FOV
        for j, comp in enumerate(COMPONENTS):
            ax = axes[0, col0 + j]
            if run["has_pred_raw"]:
                im = ax.imshow(
                    run["pred_raw"][comp].T,
                    origin="upper", cmap=jet_cmap, vmin=-corr_max, vmax=corr_max,
                )
                ax.set_title(f"{label} CNN {comp}")
            else:
                ax.text(0.5, 0.5, f"CNN {comp}\nnot available",
                        transform=ax.transAxes, ha="center", va="center",
                        fontsize=10, color="white")
                ax.set_title(f"{label} CNN {comp} (missing)")

        # Row 1: CNN polyfit correction — jet, full FOV
        for j, comp in enumerate(COMPONENTS):
            ax = axes[1, col0 + j]
            if run["has_pred_poly"]:
                im = ax.imshow(
                    run["pred_poly"][comp].T,
                    origin="upper", cmap=jet_cmap, vmin=-corr_max, vmax=corr_max,
                )
                ax.set_title(f"{label} Polyfit {comp}")
            else:
                ax.text(0.5, 0.5, f"Polyfit {comp}\nnot available",
                        transform=ax.transAxes, ha="center", va="center",
                        fontsize=10, color="white")
                ax.set_title(f"{label} Polyfit {comp} (missing)")

        # Row 2: Raw manual correction (corrected − uncorrected) — jet (same data)
        for j, comp in enumerate(COMPONENTS):
            ax = axes[2, col0 + j]
            if has_diff:
                im = ax.imshow(
                    diff[comp].T,
                    origin="upper", cmap=jet_cmap, vmin=-corr_max, vmax=corr_max,
                )
                ax.set_title(f"Man. Corr−Uncorr {comp}")
            else:
                ax.text(0.5, 0.5, f"Man. Corr−Uncorr {comp}\nnot available",
                        transform=ax.transAxes, ha="center", va="center",
                        fontsize=10, color="white")
                ax.set_title(f"Man. Corr−Uncorr {comp} (missing)")

        # Row 3: Manual polyfit correction — jet, full FOV (same data)
        for j, comp in enumerate(COMPONENTS):
            ax = axes[3, col0 + j]
            if has_manual_poly:
                im = ax.imshow(
                    manual_poly[comp].T,
                    origin="upper", cmap=jet_cmap, vmin=-corr_max, vmax=corr_max,
                )
                ax.set_title(f"Man. Polyfit {comp}")
            else:
                ax.text(0.5, 0.5, f"Man. Polyfit {comp}\nnot available",
                        transform=ax.transAxes, ha="center", va="center",
                        fontsize=10, color="white")
                ax.set_title(f"Man. Polyfit {comp} (missing)")

        # Row 4: Manual polyfit − CNN polyfit on mediastinum — RdBu_r
        for j, comp in enumerate(COMPONENTS):
            ax = axes[4, col0 + j]
            if run["has_corr_residual"]:
                im = ax.imshow(
                    apply_mask(run["corr_residual"][comp], mediastinum).T,
                    origin="upper", cmap=resid_cmap, vmin=-corr_max, vmax=corr_max,
                )
                ax.set_title(f"{label} Poly Resid {comp}")
            else:
                ax.text(0.5, 0.5, f"Poly Resid {comp}\nnot available",
                        transform=ax.transAxes, ha="center", va="center",
                        fontsize=10, color="white")
                ax.set_title(f"{label} Poly Resid {comp} (missing)")

        # Row 5: CNN corrected velocity — RdBu_r, tissue masked
        for j, comp in enumerate(COMPONENTS):
            ax = axes[5, col0 + j]
            if run["has_cnn_corr"]:
                im = ax.imshow(
                    apply_mask(run["cnn_corr"][comp], tissue).T,
                    origin="upper", cmap=vel_cmap, vmin=-vel_max, vmax=vel_max,
                )
                ax.set_title(f"{label} CNN Corr. {comp}")
            else:
                ax.text(0.5, 0.5, f"CNN Corr. {comp}\nnot available",
                        transform=ax.transAxes, ha="center", va="center",
                        fontsize=10, color="white")
                ax.set_title(f"{label} CNN Corr. {comp} (missing)")

        # Row 6: Manually corrected velocity — RdBu_r, tissue masked (same data)
        for j, comp in enumerate(COMPONENTS):
            ax = axes[6, col0 + j]
            if corr[comp] is not None:
                im = ax.imshow(
                    apply_mask(corr[comp], tissue).T,
                    origin="upper", cmap=vel_cmap, vmin=-vel_max, vmax=vel_max,
                )
                ax.set_title(f"Man. Corr. {comp}")
            else:
                ax.text(0.5, 0.5, f"Man. Corr. {comp}\nnot available",
                        transform=ax.transAxes, ha="center", va="center",
                        fontsize=10, color="white")
                ax.set_title(f"Man. Corr. {comp} (missing)")

        # Row 7: Uncorrected velocity — RdBu_r, tissue masked (same data)
        for j, comp in enumerate(COMPONENTS):
            ax = axes[7, col0 + j]
            if uncorr[comp] is not None:
                im = ax.imshow(
                    apply_mask(uncorr[comp], tissue).T,
                    origin="upper", cmap=vel_cmap, vmin=-vel_max, vmax=vel_max,
                )
                ax.set_title(f"Uncorr. {comp}")
            else:
                ax.text(0.5, 0.5, f"Uncorr. {comp}\nnot available",
                        transform=ax.transAxes, ha="center", va="center",
                        fontsize=10, color="white")
                ax.set_title(f"Uncorr. {comp} (missing)")

        # Row 8: Magnitude | Mask | Used Pixels (same data)
        axes[8, col0 + 0].imshow(mag_slice.T, origin="upper", cmap="gray")
        axes[8, col0 + 0].set_title("Mag")
        axes[8, col0 + 1].imshow(tissue.T.astype(float), origin="upper", cmap="gray")
        axes[8, col0 + 1].set_title("Mask")
        axes[8, col0 + 2].imshow(apply_mask(mag_slice, tissue).T, origin="upper", cmap="RdBu_r")
        axes[8, col0 + 2].set_title("Used Pixels")

    for ax in axes.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_facecolor(bg_color)

    return fig

In [12]:
errors = []

for idx, row in patients_df.iterrows():
    pid = row["patient_id"]
    split = row["split"]
    patient_dir = PATIENT_DATA_DIR / pid
    ds_root = patient_dir / "nifti" / DOWNSAMPLED_FOLDER

    if not ds_root.exists():
        print(f"[SKIP] {pid}: downsampled dir not found")
        errors.append((pid, split, "missing_dir"))
        continue

    n_mag_frames = len(list((ds_root / "4d_flow_mag").glob("*.nii.gz")))
    frame = min(FRAME_INDEX, n_mag_frames - 1)

    inf_root_a = INFERENCE_DIR_A / pid
    inf_root_b = INFERENCE_DIR_B / pid

    try:
        fig = make_comparison_figure(
            pid, split, frame, ds_root, patient_dir,
            inf_root_a, inf_root_b,
            run_a_label=RUN_A_NAME, run_b_label=RUN_B_NAME,
        )
        out_path = OUTPUT_DIR / f"{split}_{pid}.png"
        fig.savefig(out_path, dpi=120, bbox_inches="tight")
        plt.close(fig)
        print(f"[OK]   {pid} ({split}) -> {out_path.name}")
    except Exception as e:
        print(f"[ERR]  {pid} ({split}): {e}")
        errors.append((pid, split, str(e)))

print(f"\nDone. {len(patients_df) - len(errors)} succeeded, {len(errors)} errors.")
if errors:
    print("Errors:")
    for pid, split, msg in errors:
        print(f"  {pid} ({split}): {msg}")

[OK]   Balboloop (test) -> test_Balboloop.png
[OK]   Biswifo (test) -> test_Biswifo.png
[OK]   Bomatog (test) -> test_Bomatog.png
[OK]   Boochuto (test) -> test_Boochuto.png
[OK]   Boumorim (test) -> test_Boumorim.png
[OK]   Bovutou (test) -> test_Bovutou.png
[OK]   Cadotueg (test) -> test_Cadotueg.png
[OK]   Detodu (test) -> test_Detodu.png
[OK]   Diecudey (test) -> test_Diecudey.png
[OK]   Diepami (test) -> test_Diepami.png
[OK]   Diequipi (test) -> test_Diequipi.png
[OK]   Dithigog (test) -> test_Dithigog.png
[OK]   Dublafer (test) -> test_Dublafer.png
[OK]   Dujomal (test) -> test_Dujomal.png
[OK]   Elagieg (test) -> test_Elagieg.png
[OK]   Golotag (test) -> test_Golotag.png
[OK]   Grequafie (test) -> test_Grequafie.png
[OK]   Gueshifa (test) -> test_Gueshifa.png
[OK]   Kuquelok (test) -> test_Kuquelok.png
[OK]   Oduskueb (test) -> test_Oduskueb.png
[OK]   Quetode (test) -> test_Quetode.png
[OK]   Runusath (test) -> test_Runusath.png
[OK]   Sepigoo (test) -> test_Sepigoo.png
[OK]  